# 01 - Reading and editing an MCNP input

**~30 minutes.**



In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

## 1. Open the model

`Input.from_input` parses the file and gives you back an `Input` object.

You may see a **warning about diagnostics**. That is normal: the parser is telling you it found
things it did not fully understand (a legacy comment style, an unusual card). It is not an
error, and the model is still complete. The full list is in `inp._model.diagnostics` if you are
curious.

In [ ]:
from f4enix.input.MCNPinput import Input

inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")
inp

In [ ]:
# the title card and the comment block at the top of the file
print(inp.header)

In [ ]:
# Diagnostics can be checked accessing the migjorn model stored under ._model
# for diagnostic in inp._model.diagnostics:
#     print(f'severity={diagnostic.severity}, message={diagnostic.message}')

## 2. What an `Input` is made of

Cells, surfaces, transformations and "everything else" (the data cards) are exposed as objects
that behave like **dictionaries**. The keys are strings.

In [ ]:
print("cells          :", len(inp.cells))
print("surfaces       :", len(inp.surfs))
print("transformations:", len(inp.transformations))
print("materials      :", len(inp.mat_section))

In [ ]:
# printing a cell gives you back its card, exactly as MCNP would read it
print(inp.cells["2"])

A cell is not just text: the interesting quantities are attributes you can read **and write**.

In [ ]:
cell = inp.cells["2"]

print("id       :", cell.id)
print("material :", cell.material)
print("density  :", cell.density)
print("universe :", cell.universe)
print("fill     :", cell.fill)
print("is void  :", cell.is_void)
print("geometry :", cell.geometry_text)

In [ ]:
# surfaces work the same way
surf = inp.surfs["60"]
print(surf)
# surf is a migjorn Surface object
print("kind:", surf.mnemonic, "| coefficients:", surf.coeffs)

`other_data` holds every card that is not a cell, a surface, a transformation or a material:
the source, the tallies, the physics options, `NPS`...

These cards are parsed in a more general so their attributes and methods are more limited.

In [ ]:
print(list(inp.other_data.keys()))

In [ ]:
tally = inp.other_data["f84"]
print(tally)
print('name:', tally.name, '\nparticle:', tally.particle, '\nvalues:', tally.values)


Two methods are also provided to get all tallies IDs without you having to filter the keys yourself.

In [ ]:
print("F tallies:", inp.tally_keys)
print("FMESHes  :", inp.fmesh_keys)

## 3. A table of all the cells

`get_cells_summary()` returns a **DataFrame**: one row per cell, with its material, density,
universe and fill. This is usually the fastest way to get a feel for a model you do not know.

In [ ]:
summary = inp.get_cells_summary()
summary.head(10)

Because it is a DataFrame, you can answer questions with one line. How many cells per material?

In [ ]:
summary["material"].value_counts().sort_index()

The model is dominated by material 13 (the magnet). Which cells are **void**?

In [ ]:
void_cells = summary[summary["material"] == 0]
print("void cells:", list(void_cells.index))

And which cells use the water, material 7? Remember the pattern:
`table[table["column"] == value]`.

In [ ]:
water_cells = list(summary[summary["material"] == 7].index)
print(len(water_cells), "water cells:", water_cells)

## 4. From a material to its cells, and back

`get_cells_by_matID` does the filtering above for you and returns the actual cell objects.

In [ ]:
cells_m7 = inp.get_cells_by_matID(7)
print("cells with material 7:", list(cells_m7.keys()))

# and here is one of them
print(cells_m7["34"])

`get_materials_subset` goes the other way: give it material names, get a material section back.
It is what you use to build a reduced model that only carries the materials it needs.

In [ ]:
subset = inp.get_materials_subset(["M7", "M13"])
print([m.name for m in subset])

## 5. Density check

A classic QA question on an inherited model: *are the densities sensible?*
`get_densities_range` converts every cell density to g/cc and reports the range used by each
material. If a material shows two very different values, someone probably made a typo.

In [ ]:
inp.get_densities_range()

## 6. Carving out a sub-model

Cells 2 to 21 are the **inboard magnet**. Let's extract them into a standalone input.

`extract_cells` collects the requested cells **plus every surface, or additional cells
they need**, and returns a brand new `Input`. All datacards (including materials and transformations) are passed to this new input.

In [ ]:
magnet_cells = list(range(2, 22))

inboard = inp.extract_cells(magnet_cells)

print("cells in the extracted model:", list(inboard.cells.keys()))
print("surfaces                    :", len(inboard.surfs))

There is a catch, and it is a useful one to know: `extract_cells` also **copies all the data
cards** - source, tallies, physics. Those tallies still refer to cells that no longer exist, so
the file would not run as it is. If needed, they can be removed easily.

In [ ]:
print("before:", inboard.tally_keys)

inboard.remove_tallies()          # no argument -> removes them all
inboard.remove_sdef()             # remove the source

print("after :", inboard.tally_keys)

The materials were copied over as well - all 22 of them, although the magnet only uses one.
The `mat_section` can simply be replaced.

In [ ]:
print("materials carried over:", len(inboard.mat_section))

inboard.mat_section = inp.get_materials_subset(["M13"])

print("materials kept        :", [m.name for m in inboard.mat_section])

## 7. Writing it back out

`write` produces a real MCNP input. The round trip - write, read again - is the proof that
nothing was lost.

In [ ]:
inboard.write(OUT / "inboard_magnet.i")

check = Input.from_input(OUT / "inboard_magnet.i")
print("cells re-read:", list(check.cells.keys()))
print("materials    :", [m.name for m in check.mat_section])

In [ ]:
# have a look at the first lines of what we produced
print((OUT / "inboard_magnet.i").read_text()[:600])

## 8. Inserting a component: universes

The other half of the job is the opposite operation - putting a detailed component **inside**
an existing cell, using an MCNP universe.

We switch to a deliberately tiny pair of models so you can see every number:

* `Sphere.i` - three cells, three spherical surfaces
* `filler.i` - a single cell, the component we want to insert

In [ ]:
sphere = Input.from_input(DATA / "sphere" / "Sphere.i")
filler = Input.from_input(DATA / "sphere" / "filler.i")

print("sphere cells   :", list(sphere.cells.keys()))
print("sphere surfaces:", list(sphere.surfs.keys()))
print("filler cells   :", list(filler.cells.keys()))
print("filler surfaces:", list(filler.surfs.keys()))

Both models use cell 1 and surface 1. If we merged them right now the IDs would clash, so the
first step is always to **move the filler out of the way**.

`check_range` tells you whether a block of IDs is free.

In [ ]:
print("is 1-3 free?     ", sphere.check_range(range(1, 4), who="cell"))
print("is 101-103 free? ", sphere.check_range(range(101, 104), who="cell"))

`find_first_free_id_range` does the search for you. It looks at cell **and** surface IDs at the
same time, so the number it returns is safe for both.

In [ ]:
print("first free block of 10 IDs starts at:", sphere.find_first_free_id_range(10))

We will use a round number instead, purely so the result is easy to read. `renumber` shifts
every ID by a constant offset.

In [ ]:
filler.renumber(cells=100, surfs=100)

print("filler cells now   :", list(filler.cells.keys()))
print("filler surfaces now:", list(filler.surfs.keys()))

Now we declare that the filler cell belongs to **universe 10**. `set_param` adds - or replaces -
a keyword on a cell card.

In [ ]:
Input.set_param(filler.cells["101"], "u", 10)
print(filler.cells["101"])

`merge` brings the filler model into the sphere model. It refuses to merge if the two models
define materials with the same name, which is exactly the check you want.

In [ ]:
sphere.merge(filler)

print("cells after merge:", list(sphere.cells.keys()))
print("materials        :", [m.name for m in sphere.mat_section])

The last step is to tell cell 2 that it is filled by universe 10.

In [ ]:
Input.set_param(sphere.cells["2"], "fill", 10)
print(sphere.cells["2"])

In [ ]:
sphere.write(OUT / "sphere_filled.i")
print((OUT / "sphere_filled.i").read_text())

> **Note.** We only checked the *syntax* here. The filler sphere has a radius of 55 cm while
> cell 2 stops at 50 cm, so a real model would still need a geometry check. F4Enix edits the
> input, it does not do your neutronics for you.

Two related methods, worth knowing they exist:

* `extract_universe(u)` - the reverse of what we just did, pulls a universe out into its own model
* `delete_fill_cards()` - strips every `fill=` from the model

## 9. Editing geometry expressions

Two static helpers modify the Boolean expression of a cell: you pass them the cell object.

In [ ]:
demo = Input.from_input(DATA / "sphere" / "Sphere.i")

print("original      :", demo.cells["2"].geometry_text)

Input.hash_cell(demo.cells["2"], 1)                  # subtract cell 1
print("after hash    :", demo.cells["2"].geometry_text)

Input.add_surface(demo.cells["2"], -3, mode="intersect")
print("after surface :", demo.cells["2"].geometry_text)

`cells_union` merges several cells into one. All of them must share the same material - the
first cell in the list provides the material, density and parameters.

In [ ]:
demo2 = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")

print("before:", len(demo2.cells), "cells")
demo2.cells_union(["2", "3", "4"])     # three magnet cells, all material 13
print("after :", len(demo2.cells), "cells")
print(demo2.cells["2"])

## 10. Add a card to the input

You can easily add cards to the input as you would write MCNP entries.
Just be careful to add them to the correct group.

You can do that using the F4Enix dictionary, but, since the dictionary is is just a visualization of the underlying migjorn model, it is cleaner to directly use migjorn ``add`` methods.

In [ ]:
# Using straight migjorn syntax
inp._model.add_surface('999 so 1')
inp._model.add_cell('999 0 -999')
inp._model.add_transform('TR1 0 0 0')
inp._model.add_data_card('SDEF POS=0 0 0')

# using F4Enix syntax
inp.surfs['10000'] = '10000 so 1'
# ...

# for materials see next jupyter

---

# Exercises

## Exercise 1 - extract the outboard magnet

Suppose you want to extract the outboard magnet. You know that everything that has material 1 and 13 is part of this magnet.

Produce a standalone input file `outboard_magnet.i` in the `outputs` folder that:

1. contains only those cells (you will have to retrieve them knowing that they  must have either material 1 or 13 assigned),
2. has all its cells and surfaces **shifted by +1000**, so the IDs cannot clash with anything
   else later,
3. carries only the materials it actually needs (`M1` and `M13`),
4. has no tallies and no source left in it.


<details>
  <summary><b>Hints</b> <- Click to expand</summary>

* `extract_cells` takes a `renumber_offsets` argument: a dictionary passed straight on to
  `renumber`, e.g. `{"cells": 1000, "surfs": 1000}`;
* `remove_tallies()` and `remove_sdef()` take no arguments when you want to remove everything;
* replace `outboard.mat_section` with `inp.get_materials_subset([...])`.

</details>

In [ ]:
inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")
materials = [1, 13]

# TODO 1: Retrieve the cells to be extracted
outboard_cells = ...

# TODO 2: extract them, shifting cells and surfaces by 1000
outboard = ...

# TODO 3: remove the tallies and the source


# TODO 4: keep only the materials M1 and M13


# TODO 5: write the file to OUT / "outboard_magnet.i"

In [ ]:
# --- self-check, do not modify ---
reloaded = Input.from_input(OUT / "outboard_magnet.i")

expected_ids = {str(i) for i in range(1085, 1106)}
expected_ids.update({str(i) for i in range(1002, 1023)})
assert set(reloaded.cells.keys()) == expected_ids, (
    f"expected cells 1085-1105 and 1002-1021, got {sorted(reloaded.cells.keys())}"
)
assert reloaded.tally_keys == [], f"there are still tallies: {reloaded.tally_keys}"
assert "sdef" not in [k.lower() for k in reloaded.other_data], "the source is still there"
assert sorted(m.name for m in reloaded.mat_section) == ["M1", "M13"], (
    f"materials are {[m.name for m in reloaded.mat_section]}"
)
print("Exercise 1 correct")

## Exercise 2 - insert the filler into the sphere, twice

Repeat the universe insertion of section 8, but this time put the filler into **both** cell 1
and cell 2 of the sphere model, and use **universe 7** instead of 10.

Write the result to `outputs/sphere_double_fill.i`.


<details>
  <summary><b>Hints</b> <- Click to expand</summary>

  * the steps are always the same: renumber the filler, give its cell a `u=`, merge, then set
  `fill=` on the target cells;
  * an offset of 100 works again;
  * `Input.set_param(cell, "fill", 7)` - one call per cell you want to fill.

</details>

In [ ]:
sphere2 = Input.from_input(DATA / "sphere" / "Sphere.i")
filler2 = Input.from_input(DATA / "sphere" / "filler.i")

# TODO 1: Create the new required surfaces and cells


# TODO 2: split the cells


# TODO 3: move the filler IDs out of the way (offset 100 on cells and surfaces)


# TODO 4: put the filler cell in universe 7


# TODO 5: merge the filler into sphere2


# TODO 6: fill cells 1 and 2 with universe 7


# TODO 7: write to OUT / "sphere_double_fill.i"

In [ ]:
# --- self-check, do not modify ---
res = Input.from_input(OUT / "sphere_double_fill.i")

assert "101" in res.cells, f"the filler cell is missing, cells are {list(res.cells.keys())}"
assert "999" in res.cells, f"the new cell is missing, cells are {list(res.cells.keys())}"
assert res.cells["101"].universe == 7, f"filler universe is {res.cells['101'].universe}"
for cid in ("999", "2"):
    assert res.cells[cid].fill is not None, f"cell {cid} has no fill card"
    assert res.cells[cid].fill.universe == 7, f"cell {cid} fill is {res.cells[cid].fill.universe}"
assert res.cells["2"].geometry[-1].text == '999', f"last geometry entry for cell 2 is {res.cells['2'].geometry[-1]}"
assert res.cells["999"].geometry[-1].text == '-999', f"last geometry entry for cell 999 is {res.cells['999'].geometry[-1]}"
for cell in ["2", "999"]:
    assert res.cells[cell].geometry[-2].text == ')', f"cell {cell} was not split it seems"
print("Exercise 2 correct")

---

## What you can now do

| Question | Method |
|----------|--------|
| what is in this model? | `get_cells_summary`, `tally_keys`, `other_data.keys()` |
| which cells use material X? | `get_cells_by_matID` |
| are the densities sane? | `get_densities_range` |
| give me a small standalone piece | `extract_cells` + `remove_tallies` + `remove_sdef` |
| are these IDs free? | `check_range`, `find_first_free_id_range` |
| move a model out of the way | `renumber` |
| glue two models together | `merge` |
| put a component inside a cell | `set_param(cell, "u"/"fill", ...)` |

Next: **`02_materials_tallies_and_run_settings.ipynb`** - materials, cross-section libraries and
the cards that decide what the run actually computes.